# GCP Data Engineering — First Contact

Google Cloud's data engineering stack is built around a few core managed services that fit together cleanly: Cloud Storage is the lake layer for raw files, BigQuery is the serverless warehouse and analytics engine, Pub/Sub is the managed streaming backbone, and Dataflow is the processing layer that moves and transforms data between systems. The mental model is simple: land data, analyze data, stream events, and orchestrate movement without managing clusters.

BigQuery is the center of gravity here. It is not just a query engine over files — it is a serverless data warehouse that stores and optimizes data internally, executes SQL directly, and handles scaling, storage layout, pruning, and performance tuning behind the scenes. Unlike a traditional cluster-based warehouse, there is no infrastructure to size, patch, or tune. You load data and query it.

For a Citi-style telemetry stack, GCP's edge is operational simplicity. BigQuery is fully managed — no Glue catalog needed, no Parquet format decisions. Load CSV, BigQuery handles storage optimization internally. Compared with Athena, the key distinction is that BigQuery stores and manages the warehouse data itself, while Athena primarily queries data sitting in S3. In this notebook, we use Cloud Storage as the lake, BigQuery as the warehouse, and Pub/Sub as the streaming layer.

```text
[Postgres] → [GCS] → [BigQuery] ← [Pub/Sub → Dataflow]
```

In [1]:
# Packages pre-installed: google-cloud-bigquery google-cloud-storage google-cloud-pubsub psycopg2-binary pandas pyarrow
import json
import os

import pandas as pd
import psycopg2
from google.api_core.exceptions import AlreadyExists, Conflict, NotFound
from google.cloud import bigquery
from google.cloud import storage
from google.cloud import pubsub_v1

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "D:/Workspace/Technologies/_setup/gcp_key.json"

PROJECT_ID     = "citi-de-learning"
DATASET_ID     = "citi_telemetry"
BUCKET_NAME    = f"citi-telemetry-gcs-{PROJECT_ID}"
REGION         = "us-central1"
TOPIC_ID       = "citi-alerts"
SUBSCRIPTION_ID = "citi-alerts-sub"

storage_client   = storage.Client(project=PROJECT_ID)
bigquery_client  = bigquery.Client(project=PROJECT_ID)
publisher        = pubsub_v1.PublisherClient()
subscriber       = pubsub_v1.SubscriberClient()

print(f"GCP Project: {PROJECT_ID}")
print("Clients initialized.")

GCP Project: citi-de-learning
Clients initialized.


Create the GCS bucket and BigQuery dataset

In [2]:
bucket = storage_client.bucket(BUCKET_NAME)

try:
    bucket = storage_client.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Created bucket: {bucket.name}")
except Conflict:
    bucket = storage_client.bucket(BUCKET_NAME)
    print(f"Bucket already exists: {bucket.name}")

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = REGION
dataset = bigquery_client.create_dataset(dataset_ref, exists_ok=True)

print(f"GCS bucket {BUCKET_NAME} ready. BigQuery dataset {DATASET_ID} ready.")


Bucket already exists: citi-telemetry-gcs-citi-de-learning


GCS bucket citi-telemetry-gcs-citi-de-learning ready. BigQuery dataset citi_telemetry ready.


## Loading Data into BigQuery

BigQuery accepts CSV, JSON, Parquet, and Avro as common input formats. In this notebook, we use a practical path for engineering workflows: load data from Postgres into pandas DataFrames, then use the native BigQuery client to load those DataFrames directly into managed BigQuery tables. This keeps the notebook simple while still reflecting a production-friendly ingestion path.

In [3]:
pg_conn = psycopg2.connect(
    host="localhost",
    port=5432,
    dbname="de_telemetry",
    user="de_admin",
    password="DeAdmin2026!"
)

endpoints_query = "SELECT endpoint_id, name, region, status, category FROM endpoints"
alerts_query = "SELECT alert_id, endpoint_id, severity, message, created_at FROM alerts"

endpoints_df = pd.read_sql(endpoints_query, pg_conn)
alerts_df = pd.read_sql(alerts_query, pg_conn)

pg_conn.close()

endpoints_table_id = f"{PROJECT_ID}.{DATASET_ID}.endpoints"
alerts_table_id = f"{PROJECT_ID}.{DATASET_ID}.alerts"

load_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)

endpoints_job = bigquery_client.load_table_from_dataframe(
    endpoints_df,
    endpoints_table_id,
    job_config=load_config,
)
alerts_job = bigquery_client.load_table_from_dataframe(
    alerts_df,
    alerts_table_id,
    job_config=load_config,
)

endpoints_job.result()
alerts_job.result()

print(f"Loaded {len(endpoints_df)} endpoints and {len(alerts_df)} alerts to BigQuery")


C:\Users\shareuser\AppData\Local\Temp\ipykernel_62288\381432898.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  endpoints_df = pd.read_sql(endpoints_query, pg_conn)
C:\Users\shareuser\AppData\Local\Temp\ipykernel_62288\381432898.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  alerts_df = pd.read_sql(alerts_query, pg_conn)


Loaded 10000 endpoints and 25000 alerts to BigQuery


## BigQuery — Serverless SQL at Scale

BigQuery uses GoogleSQL (Standard SQL), scales without cluster management, and prices on demand by data scanned or through reserved slots. The common on-demand model is approximately $5 per TB scanned, and the BigQuery free tier includes up to 1 TB of query data processed per month. For many demos, labs, and moderate analytical workloads, that makes it a strong first-contact warehouse for engineers.

In [4]:
def run_query(sql: str, label: str) -> pd.DataFrame:
    print(f"\n=== {label} ===")
    df = bigquery_client.query(sql).result().to_dataframe()
    if df.empty:
        print("No rows returned.")
    else:
        print(df.to_string(index=False))
    return df

query_1 = f'''
SELECT severity, COUNT(*) as cnt
FROM `{PROJECT_ID}.{DATASET_ID}.alerts`
GROUP BY severity
ORDER BY cnt DESC
'''

query_2 = f'''
SELECT e.region, a.severity, COUNT(*) as alert_count
FROM `{PROJECT_ID}.{DATASET_ID}.alerts` a
JOIN `{PROJECT_ID}.{DATASET_ID}.endpoints` e
  ON a.endpoint_id = e.endpoint_id
GROUP BY e.region, a.severity
ORDER BY alert_count DESC
LIMIT 20
'''

query_3 = f'''
SELECT e.name, e.region, COUNT(*) as alert_count
FROM `{PROJECT_ID}.{DATASET_ID}.alerts` a
JOIN `{PROJECT_ID}.{DATASET_ID}.endpoints` e
  ON a.endpoint_id = e.endpoint_id
GROUP BY e.name, e.region
ORDER BY alert_count DESC
LIMIT 10
'''

q1_df = run_query(query_1, "Query 1 — Alert counts by severity")
q2_df = run_query(query_2, "Query 2 — Regional alert summary with endpoint join")
q3_df = run_query(query_3, "Query 3 — Top 10 most alerted endpoints")



=== Query 1 — Alert counts by severity ===


severity  cnt
    HIGH 6313
     LOW 6254
  MEDIUM 6248
CRITICAL 6185

=== Query 2 — Regional alert summary with endpoint join ===


region severity  alert_count
  SNG1   MEDIUM         1656
  SNG1      LOW         1640
  SNG1 CRITICAL         1635
  NYC1     HIGH         1589
  SNG1     HIGH         1588
  NYC1   MEDIUM         1580
  NYC1      LOW         1580
  NYC2     HIGH         1579
  NYC2   MEDIUM         1559
  LON1     HIGH         1557
  NYC1 CRITICAL         1537
  NYC2      LOW         1532
  LON1 CRITICAL         1509
  NYC2 CRITICAL         1504
  LON1      LOW         1502
  LON1   MEDIUM         1453

=== Query 3 — Top 10 most alerted endpoints ===


                   name region  alert_count
srv-04452.citi.internal   NYC1           12
srv-07578.citi.internal   NYC1           11
srv-02253.citi.internal   SNG1           10
srv-03610.citi.internal   NYC2           10
srv-03423.citi.internal   NYC1           10
srv-01617.citi.internal   SNG1           10
srv-02223.citi.internal   NYC1            9
srv-02944.citi.internal   LON1            9
srv-07573.citi.internal   LON1            9
srv-09621.citi.internal   NYC2            9


## Pub/Sub — GCP's Message Queue

Pub/Sub is Google's managed messaging backbone and is the closest high-level analogue to a managed Kafka-style event transport in GCP. It revolves around topics and subscriptions, supports both push and pull consumption models, and is globally managed by default. You do not manage brokers, partitions, or replication manually — that operational complexity is abstracted away by the service.

In [5]:
topic_path = publisher.topic_path(PROJECT_ID, TOPIC_ID)
sub_path = subscriber.subscription_path(PROJECT_ID, SUBSCRIPTION_ID)

try:
    publisher.create_topic(request={"name": topic_path})
    print(f"Created topic: {topic_path}")
except AlreadyExists:
    print(f"Topic already exists: {topic_path}")

try:
    subscriber.create_subscription(request={"name": sub_path, "topic": topic_path})
    print(f"Created subscription: {sub_path}")
except AlreadyExists:
    print(f"Subscription already exists: {sub_path}")

sample_alerts = []
for i in range(10):
    row = alerts_df.iloc[i]
    payload = {
        "alert_id": int(row["alert_id"]),
        "endpoint_id": int(row["endpoint_id"]),
        "severity": str(row["severity"]),
        "message": str(row["message"]),
        "created_at": row["created_at"].isoformat() if hasattr(row["created_at"], "isoformat") else str(row["created_at"]),
    }
    sample_alerts.append(payload)

publish_futures = []
for payload in sample_alerts:
    data = json.dumps(payload).encode("utf-8")
    publish_futures.append(publisher.publish(topic_path, data=data))

published_message_ids = [future.result() for future in publish_futures]
print(f"Published {len(published_message_ids)} messages to citi-alerts topic")


Created topic: projects/citi-de-learning/topics/citi-alerts


Created subscription: projects/citi-de-learning/subscriptions/citi-alerts-sub
Published 10 messages to citi-alerts topic


In [6]:
response = subscriber.pull(
    request={
        "subscription": sub_path,
        "max_messages": 10,
    }
)

acked_ids = []
for received in response.received_messages:
    message_data = json.loads(received.message.data.decode("utf-8"))
    print(message_data)
    acked_ids.append(received.ack_id)

if acked_ids:
    subscriber.acknowledge(request={"subscription": sub_path, "ack_ids": acked_ids})

print(f"Pulled and acked {len(acked_ids)} messages")


{'alert_id': 1, 'endpoint_id': 7416, 'severity': 'CRITICAL', 'message': 'Health check failed: HTTP 503 from email-37.stevens.com', 'created_at': '2026-01-13T11:40:27.063609+00:00'}
{'alert_id': 2, 'endpoint_id': 3694, 'severity': 'CRITICAL', 'message': 'TCP connection pool exhausted on port 5432', 'created_at': '2026-01-24T11:42:23.063609+00:00'}
{'alert_id': 3, 'endpoint_id': 5828, 'severity': 'HIGH', 'message': 'Memory usage at 95% — potential OOM imminent', 'created_at': '2026-01-16T11:11:06.063609+00:00'}
{'alert_id': 4, 'endpoint_id': 1986, 'severity': 'MEDIUM', 'message': 'Network throughput dropped below SLA threshold', 'created_at': '2026-01-01T06:01:32.063609+00:00'}
{'alert_id': 5, 'endpoint_id': 4807, 'severity': 'CRITICAL', 'message': 'SSL certificate expires in 7 days', 'created_at': '2026-01-12T16:04:36.063609+00:00'}
{'alert_id': 6, 'endpoint_id': 907, 'severity': 'HIGH', 'message': 'Network throughput dropped below SLA threshold', 'created_at': '2026-01-26T13:42:25.0636

## BigQuery vs Athena — Staff DE Decision Matrix

| Feature | BigQuery | Athena |
|---------|---------|--------|
| Storage | Managed (Capacitor format) | S3 (you choose format) |
| Pricing model | $5/TB scanned (on-demand) or flat slots | $5/TB scanned |
| Schema catalog | Built-in | Glue Data Catalog |
| Streaming inserts | Native (BigQuery Storage Write API) | Not supported natively |
| ML integration | BigQuery ML (SQL to train models) | SageMaker (separate) |
| Multi-cloud | GCP only | AWS only |
| Best for | GCP shops, streaming analytics, ML SQL | AWS shops, existing S3 data lake |

In [7]:
# Clean up Pub/Sub resources
try:
    subscriber.delete_subscription(request={"subscription": sub_path})
    print(f"Deleted subscription: {sub_path}")
except NotFound:
    print(f"Subscription not found: {sub_path}")

try:
    publisher.delete_topic(request={"topic": topic_path})
    print(f"Deleted topic: {topic_path}")
except NotFound:
    print(f"Topic not found: {topic_path}")

# Clean up BigQuery dataset and its contents
try:
    bigquery_client.delete_dataset(
        f"{PROJECT_ID}.{DATASET_ID}",
        delete_contents=True,
        not_found_ok=True,
    )
    print(f"Deleted BigQuery dataset: {PROJECT_ID}.{DATASET_ID}")
except NotFound:
    print(f"BigQuery dataset not found: {PROJECT_ID}.{DATASET_ID}")

# Clean up GCS bucket and all objects
bucket = storage_client.bucket(BUCKET_NAME)
try:
    blobs = list(storage_client.list_blobs(BUCKET_NAME))
    for blob in blobs:
        blob.delete()
    bucket.delete()
    print(f"Deleted bucket: {BUCKET_NAME}")
except NotFound:
    print(f"GCS bucket not found: {BUCKET_NAME}")

print("Clean up complete — all GCP resources deleted")


Deleted subscription: projects/citi-de-learning/subscriptions/citi-alerts-sub
Deleted topic: projects/citi-de-learning/topics/citi-alerts


Deleted BigQuery dataset: citi-de-learning.citi_telemetry


Deleted bucket: citi-telemetry-gcs-citi-de-learning
Clean up complete — all GCP resources deleted


## What Just Happened

- BigQuery dataset created
- Postgres → BigQuery loaded
- 3 SQL queries run
- Pub/Sub topic + subscription created
- 10 messages published and consumed
- cleaned up

Citi tie-in: GCP alternative to AWS: BigQuery replaces Athena + Redshift, Pub/Sub replaces Kinesis, Dataflow replaces EMR. The SQL is identical — the infrastructure is different.

Next: Run azure_de_intro.ipynb for the third cloud, then multicloud_concepts.md.